In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        if filename.endswith('.npy'):
            print("FOUND IT! Paste this exact path into your code:")
            print(os.path.join(dirname, filename))

FOUND IT! Paste this exact path into your code:
/kaggle/input/datasets/giljorge/a1-results/master_vocab_centroids.npy


In [ ]:
"""
Experiment D1 -- Eavesdropper Inversion Resistance in Dense AI-AI Channels
============================================================================

Programme: extends Programme 1 (channel constraints) with a third axis --
PRIVACY -- alongside the compression/resilience axes already characterised
in Experiment C1 (Shannon Source/Channel Pareto Frontier).

Setup
-----
A Lewis-style referential game with three parties:

  SENDER     : embedding(target concept) -> message (k-dim vector or
               Gumbel-Softmax discrete code)
  RECEIVER   : message -> concept-identification logits (task: pick the
               target concept out of a candidate pool)
  ADVERSARY  : intercepted message -> reconstructed embedding. Evaluated by
               nearest-neighbour retrieval (MRR / top-1) against the full
               concept bank, INCLUDING held-out concepts never seen during
               training. This mirrors the zero-shot / generalising inversion
               attacks (cf. Zero2Text, BeamClean) that defeat naive noise
               defenses in the literature.

Channel variants (mirrors C1's protocol list)
----------------------------------------------
  dense_k         : continuous tanh-bounded k-dim vector, k in {2,4,8,16,32,64}
  gumbel_tau      : discrete Gumbel-Softmax code, vocab V, temperature tau
  dense_k+noise   : dense_k with additive Gaussian noise (sigma sweep)
  dense_k+adv     : dense_k with adversarial training against a live
                    adversary snapshot (privacy-aware sender)

Outputs
-------
  results/exp_d1_inversion_resistance.csv   -- one row per configuration:
      protocol, k_or_vocab, defense, sigma, lambda_adv,
      task_accuracy, adv_mrr_heldout, adv_top1_heldout,
      adv_mrr_train, adv_top1_train
  results/exp_d1_pareto.png                 -- utility (task accuracy) vs
                                                privacy (1 - adv_mrr_heldout)

Usage
-----
  python exp_d1_inversion_resistance.py --quick        # fast smoke test
  python exp_d1_inversion_resistance.py                # full sweep
  python exp_d1_inversion_resistance.py --emb path.npy # plug in real LaBSE
"""

import argparse
import csv
import json
import os
import copy

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ---------------------------------------------------------------------------
# Data
# ---------------------------------------------------------------------------

def load_concept_embeddings(path=None, n_concepts=160, dim=768, seed=0):
    """Load real (e.g. LaBSE) embeddings from .npy, or synthesise a clustered
    embedding bank that mimics LaBSE's geometry (concepts cluster into
    semantic groups, as found in Exp 4/6/7)."""
    if path is not None:
        emb = np.load(path).astype(np.float32)
        emb = emb / np.linalg.norm(emb, axis=1, keepdims=True)
        return emb

    rng = np.random.default_rng(seed)
    n_clusters = max(4, n_concepts // 8)
    centroids = rng.normal(size=(n_clusters, dim)).astype(np.float32)
    centroids /= np.linalg.norm(centroids, axis=1, keepdims=True)
    assign = rng.integers(0, n_clusters, size=n_concepts)
    emb = centroids[assign] + 0.15 * rng.normal(size=(n_concepts, dim)).astype(np.float32)
    emb /= np.linalg.norm(emb, axis=1, keepdims=True)
    return emb


# ---------------------------------------------------------------------------
# Models
# ---------------------------------------------------------------------------

class Sender(nn.Module):
    """Continuous dense channel: embedding -> k-dim tanh-bounded message."""

    def __init__(self, in_dim, k, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, k), nn.Tanh(),
        )

    def forward(self, x):
        return self.net(x)


class GumbelSender(nn.Module):
    """Discrete channel: embedding -> Gumbel-Softmax code over a vocab of
    size V, length L (default L=1 single-symbol code for simplicity)."""

    def __init__(self, in_dim, vocab, length=1, hidden=256):
        super().__init__()
        self.vocab = vocab
        self.length = length
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, vocab * length),
        )

    def forward(self, x, tau, hard=False):
        logits = self.net(x).view(-1, self.length, self.vocab)
        y = F.gumbel_softmax(logits, tau=tau, hard=hard, dim=-1)
        return y.view(x.shape[0], -1)  # flatten to (B, vocab*length)


class Receiver(nn.Module):
    """message -> embedding-space prediction, scored against candidates via
    dot product (standard Lewis-game decoder)."""

    def __init__(self, msg_dim, out_dim, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(msg_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, out_dim),
        )

    def forward(self, m):
        return self.net(m)


class Adversary(nn.Module):
    """Eavesdropper: intercepted message -> reconstructed embedding.
    Trained purely on (message, true_embedding) pairs from TRAIN concepts;
    evaluated by retrieval against TRAIN+HELDOUT concept banks."""

    def __init__(self, msg_dim, out_dim, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(msg_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, out_dim),
        )

    def forward(self, m):
        return self.net(m)


# ---------------------------------------------------------------------------
# Retrieval metrics
# ---------------------------------------------------------------------------

def retrieval_metrics(recon, bank, true_idx):
    """recon: (N, D) reconstructed embeddings (unit-normalised inside).
    bank:   (M, D) concept embedding bank to retrieve against.
    true_idx: (N,) index into `bank` of the true concept for each recon row.
    Returns (mrr, top1_acc)."""
    recon_n = F.normalize(recon, dim=-1)
    bank_n = F.normalize(bank, dim=-1)
    sims = recon_n @ bank_n.T  # (N, M)
    ranks = sims.argsort(dim=-1, descending=True)
    N = recon.shape[0]
    rr = torch.zeros(N, device=recon.device)
    top1 = torch.zeros(N, device=recon.device)
    for i in range(N):
        pos = (ranks[i] == true_idx[i]).nonzero(as_tuple=True)[0].item()
        rr[i] = 1.0 / (pos + 1)
        top1[i] = float(pos == 0)
    return rr.mean().item(), top1.mean().item()


# ---------------------------------------------------------------------------
# Training loop for one configuration
# ---------------------------------------------------------------------------

def run_config(emb_train, emb_heldout, cfg, n_epochs=300, batch_size=64,
                lr=1e-3, verbose=False):
    """emb_train, emb_heldout: (N_train, D) / (N_heldout, D) torch tensors.
    cfg: dict describing the channel + defense, see build_sweep_configs()."""

    D = emb_train.shape[1]
    N = emb_train.shape[0]

    protocol = cfg["protocol"]
    if protocol == "dense":
        sender = Sender(D, cfg["k"]).to(DEVICE)
        msg_dim = cfg["k"]
    elif protocol == "gumbel":
        sender = GumbelSender(D, cfg["vocab"], length=cfg.get("length", 1)).to(DEVICE)
        msg_dim = cfg["vocab"] * cfg.get("length", 1)
    else:
        raise ValueError(protocol)

    receiver = Receiver(msg_dim, D).to(DEVICE)
    adversary = Adversary(msg_dim, D).to(DEVICE)
    adv_snapshot = copy.deepcopy(adversary).to(DEVICE)
    for p in adv_snapshot.parameters():
        p.requires_grad_(False)

    sender_opt = torch.optim.Adam(
        list(sender.parameters()) + list(receiver.parameters()), lr=lr)
    adv_opt = torch.optim.Adam(adversary.parameters(), lr=lr)

    sigma = cfg.get("sigma", 0.0)
    lambda_adv = cfg.get("lambda_adv", 0.0)
    snapshot_every = 5

    for epoch in range(n_epochs):
        perm = torch.randperm(N, device=DEVICE)
        for start in range(0, N, batch_size):
            idx = perm[start:start + batch_size]
            target_emb = emb_train[idx]

            # --- sender -> message ---
            if protocol == "dense":
                msg = sender(target_emb)
            else:
                tau = cfg.get("tau", 1.0)
                msg = sender(target_emb, tau=tau, hard=False)

            if sigma > 0:
                msg_t = msg + sigma * torch.randn_like(msg)
            else:
                msg_t = msg

            # --- receiver: predict embedding, score against full train bank ---
            pred = receiver(msg_t)
            logits = F.normalize(pred, dim=-1) @ F.normalize(emb_train, dim=-1).T
            logits = logits * 10.0  # temperature for softmax sharpness
            task_loss = F.cross_entropy(logits, idx)

            total_loss = task_loss
            if lambda_adv > 0:
                # privacy term: sender wants a FROZEN snapshot of the
                # adversary to reconstruct poorly -> maximise its MSE
                adv_pred_for_sender = adv_snapshot(msg_t)
                adv_mse_for_sender = F.mse_loss(adv_pred_for_sender, target_emb)
                total_loss = task_loss - lambda_adv * adv_mse_for_sender

            sender_opt.zero_grad()
            total_loss.backward()
            sender_opt.step()

            # --- adversary: train to reconstruct from detached message ---
            adv_pred = adversary(msg_t.detach())
            adv_loss = F.mse_loss(adv_pred, target_emb)
            adv_opt.zero_grad()
            adv_loss.backward()
            adv_opt.step()

        if lambda_adv > 0 and (epoch + 1) % snapshot_every == 0:
            adv_snapshot.load_state_dict(adversary.state_dict())

        if verbose and (epoch + 1) % max(1, n_epochs // 5) == 0:
            print(f"  epoch {epoch+1}/{n_epochs}  task_loss={task_loss.item():.4f}  "
                  f"adv_loss={adv_loss.item():.4f}")

    # -------------------------------------------------------------------
    # Evaluation
    # -------------------------------------------------------------------
    with torch.no_grad():
        def encode(emb):
            if protocol == "dense":
                return sender(emb)
            else:
                return sender(emb, tau=cfg.get("tau", 1.0), hard=True)

        # Task accuracy on train concepts (closed-set referential game)
        msg_train = encode(emb_train)
        if sigma > 0:
            msg_train_eval = msg_train + sigma * torch.randn_like(msg_train)
        else:
            msg_train_eval = msg_train
        pred_train = receiver(msg_train_eval)
        sims = F.normalize(pred_train, dim=-1) @ F.normalize(emb_train, dim=-1).T
        task_acc = (sims.argmax(dim=-1) == torch.arange(N, device=DEVICE)).float().mean().item()

        # Adversary inversion: train-set messages
        recon_train = adversary(msg_train_eval)
        mrr_train, top1_train = retrieval_metrics(
            recon_train, emb_train, torch.arange(N, device=DEVICE))

        # Adversary inversion: HELD-OUT concepts (never seen by anyone)
        msg_heldout = encode(emb_heldout)
        if sigma > 0:
            msg_heldout_eval = msg_heldout + sigma * torch.randn_like(msg_heldout)
        else:
            msg_heldout_eval = msg_heldout
        recon_heldout = adversary(msg_heldout_eval)
        # retrieval bank = train + heldout concepts together
        full_bank = torch.cat([emb_train, emb_heldout], dim=0)
        true_idx_heldout = torch.arange(
            N, N + emb_heldout.shape[0], device=DEVICE)
        mrr_heldout, top1_heldout = retrieval_metrics(
            recon_heldout, full_bank, true_idx_heldout)

    return {
        "task_accuracy": task_acc,
        "adv_mrr_train": mrr_train,
        "adv_top1_train": top1_train,
        "adv_mrr_heldout": mrr_heldout,
        "adv_top1_heldout": top1_heldout,
    }


# ---------------------------------------------------------------------------
# Sweep configuration
# ---------------------------------------------------------------------------

def build_sweep_configs(quick=False):
    configs = []

    ks = [2, 4, 8, 16, 32, 64] if not quick else [2, 8, 32]
    sigmas = [0.0, 0.1, 0.3, 0.5] if not quick else [0.0, 0.3]
    lambdas = [0.0, 1.0, 5.0] if not quick else [0.0, 2.0]

    # 1. dense_k, no defense
    for k in ks:
        configs.append({"protocol": "dense", "k": k, "defense": "none",
                         "sigma": 0.0, "lambda_adv": 0.0})

    # 2. dense_k + noise sweep (fix k=8 and k=32 as representative points)
    for k in [8, 32]:
        for sigma in sigmas:
            if sigma == 0.0:
                continue  # already covered above
            configs.append({"protocol": "dense", "k": k, "defense": f"noise_{sigma}",
                             "sigma": sigma, "lambda_adv": 0.0})

    # 3. dense_k + adversarial training
    for k in [8, 32]:
        for lam in lambdas:
            if lam == 0.0:
                continue
            configs.append({"protocol": "dense", "k": k,
                             "defense": f"adv_lambda_{lam}",
                             "sigma": 0.0, "lambda_adv": lam})

    # 4. Gumbel discrete channel (cf. Exp 1/3): vocab sweep at tau=1.0
    vocabs = [8, 32, 128] if not quick else [8, 32]
    for v in vocabs:
        configs.append({"protocol": "gumbel", "vocab": v, "length": 1,
                         "tau": 1.0, "defense": "none",
                         "sigma": 0.0, "lambda_adv": 0.0})

    return configs


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--emb", type=str, default=None,
                    help="Path to .npy concept embedding bank (e.g. LaBSE). "
                         "If omitted, a synthetic clustered embedding bank is used.")
    ap.add_argument("--n_concepts", type=int, default=160)
    ap.add_argument("--dim", type=int, default=768)
    ap.add_argument("--heldout_frac", type=float, default=0.2)
    ap.add_argument("--epochs", type=int, default=300)
    ap.add_argument("--quick", action="store_true",
                    help="Fast smoke test: fewer configs, fewer epochs, smaller bank.")
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--outdir", type=str, default="results")
    
    # We pass the arguments explicitly here for the notebook environment
    # Pass the Kaggle dataset path explicitly
    args = ap.parse_args(args=["--emb", "/kaggle/input/datasets/giljorge/a1-results/master_vocab_centroids.npy"])

    
    if args.quick:
        args.n_concepts = min(args.n_concepts, 48)
        args.epochs = min(args.epochs, 40)

    os.makedirs(args.outdir, exist_ok=True)

    emb = load_concept_embeddings(args.emb, n_concepts=args.n_concepts,
                                   dim=args.dim, seed=args.seed)
    emb = torch.tensor(emb, dtype=torch.float32, device=DEVICE)

    n = emb.shape[0]
    n_heldout = max(2, int(n * args.heldout_frac))
    g = torch.Generator(device="cpu").manual_seed(args.seed)
    perm = torch.randperm(n, generator=g)
    heldout_idx = perm[:n_heldout]
    train_idx = perm[n_heldout:]

    emb_train = emb[train_idx]
    emb_heldout = emb[heldout_idx]

    print(f"Concepts: {n} total -> {emb_train.shape[0]} train / "
          f"{emb_heldout.shape[0]} held-out, dim={emb.shape[1]}, device={DEVICE}")

    configs = build_sweep_configs(quick=args.quick)
    rows = []
    for i, cfg in enumerate(configs):
        label = cfg["protocol"]
        if cfg["protocol"] == "dense":
            label += f"_k{cfg['k']}"
        else:
            label += f"_v{cfg['vocab']}_tau{cfg.get('tau')}"
        label += f"_{cfg['defense']}"
        print(f"[{i+1}/{len(configs)}] {label}")

        metrics = run_config(emb_train, emb_heldout, cfg,
                              n_epochs=args.epochs, verbose=False)
        row = {"label": label, **{k: v for k, v in cfg.items()}, **metrics}
        rows.append(row)
        print(f"    task_acc={metrics['task_accuracy']:.3f}  "
              f"adv_mrr_heldout={metrics['adv_mrr_heldout']:.3f}  "
              f"adv_top1_heldout={metrics['adv_top1_heldout']:.3f}")

    # write CSV
    csv_path = os.path.join(args.outdir, "exp_d1_inversion_resistance.csv")
    fieldnames = sorted(set().union(*[r.keys() for r in rows]))
    with open(csv_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for r in rows:
            writer.writerow(r)
    print(f"Wrote {csv_path}")

    # write JSON for convenience too
    json_path = os.path.join(args.outdir, "exp_d1_inversion_resistance.json")
    with open(json_path, "w") as f:
        json.dump(rows, f, indent=2)

    # Pareto plot: utility (task accuracy) vs privacy (1 - adv_mrr_heldout)
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt

        fig, ax = plt.subplots(figsize=(7, 6))
        for r in rows:
            x = r["task_accuracy"]
            y = 1.0 - r["adv_mrr_heldout"]
            ax.scatter(x, y, s=40)
            ax.annotate(r["label"], (x, y), fontsize=7,
                         xytext=(3, 3), textcoords="offset points")
        ax.set_xlabel("Task accuracy (utility)")
        ax.set_ylabel("1 - adversary MRR on held-out concepts (privacy)")
        ax.set_title("Experiment D1: Utility-Privacy Pareto Frontier")
        ax.grid(alpha=0.3)
        fig.tight_layout()
        png_path = os.path.join(args.outdir, "exp_d1_pareto.png")
        fig.savefig(png_path, dpi=150)
        print(f"Wrote {png_path}")
    except ImportError:
        print("matplotlib not available, skipping plot")


if __name__ == "__main__":
    main()

if __name__ == "__main__":
    main()
